## Task 2 — Clean + Enrich Data (`data/` read-only)

Implements the data-processing requirements:
1) Clean and standardize records
2) Remove inconsistencies and duplicates
3) Merge multiple data sources into a unified dataset

Scope update (per latest instruction):
- **Training dataset** is produced for **all users**.
- Optional: you can export a single-user JSON by setting `FOCUS_CLIENT_ID` in the Config cell.

Read-only guarantee:
- Reads from `data/` only; never modifies files under `data/`.

Outputs (flat files under `artifacts/`):
- `artifacts/transactions_enriched.json` (NDJSON / JSON Lines; one JSON object per line) — all users
- Optional: `artifacts/transactions_enriched_{client_id}.json` (pretty JSON array) — if `FOCUS_CLIENT_ID` is set
- `artifacts/qa_report.json` — pipeline QA counters and assumptions

PII policy (per `PROJECT_SPEC.md`): exclude `address`, `card_number`, and `cvv` from all outputs.


## Imports

Uses `pandas` for chunked CSV reads and streaming NDJSON writes.

In [6]:
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd


## Config

This notebook cleans **all users** for training output (writes one NDJSON artifact under `artifacts/`).

In [7]:
FOCUS_CLIENT_ID = None  # [ASSUMPTION] set to an int to also export `transactions_enriched_{client_id}.json`
TRANSACTIONS_CHUNKSIZE = 250_000
QA_SAMPLE_ROWS = 5


def find_project_root(start: Path | None = None) -> Path:
    """Find project root by locating the `data/` directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing a 'data/' directory")


ROOT = find_project_root()
DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"

PATHS = {
    "transactions": DATA_DIR / "transactions.csv",
    "cards": DATA_DIR / "cards.csv",
    "users": DATA_DIR / "users.csv",
    "mcc": DATA_DIR / "mcc_codes.json",
}

OUT_ALL_NDJSON = ARTIFACTS_DIR / "transactions_enriched.json"  # NDJSON
OUT_FOCUS_JSON = (
    ARTIFACTS_DIR / f"transactions_enriched_{FOCUS_CLIENT_ID}.json"
    if FOCUS_CLIENT_ID is not None
    else None
)
OUT_QA = ARTIFACTS_DIR / "qa_report.json"

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
for k, v in PATHS.items():
    print(f"{k} -> {v}")
print("OUT_ALL_NDJSON:", OUT_ALL_NDJSON)
print("OUT_FOCUS_JSON:", OUT_FOCUS_JSON if OUT_FOCUS_JSON is not None else "(disabled)")
print("OUT_QA:", OUT_QA)


ROOT: /Users/nicholasp/Personal Coding/JHU/personal finance
DATA_DIR: /Users/nicholasp/Personal Coding/JHU/personal finance/data
ARTIFACTS_DIR: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts
transactions -> /Users/nicholasp/Personal Coding/JHU/personal finance/data/transactions.csv
cards -> /Users/nicholasp/Personal Coding/JHU/personal finance/data/cards.csv
users -> /Users/nicholasp/Personal Coding/JHU/personal finance/data/users.csv
mcc -> /Users/nicholasp/Personal Coding/JHU/personal finance/data/mcc_codes.json
OUT_ALL_NDJSON: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transactions_enriched.json
OUT_FOCUS_JSON: (disabled)
OUT_QA: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/qa_report.json


## Cleaning rules (annotated)

What gets cleaned/standardized:
- Text standardization: trim whitespace and collapse internal whitespace on key string columns.
- `amount` (currency string) → `amount_usd` (float). **Invalid amounts are dropped** (counted in QA).
- `date` (string) → `transaction_dt` (datetime). **Invalid dates are dropped** (counted in QA).
- `zip` → `zip_norm` (string, strips trailing `.0`).
- `mcc` → `mcc_code` (digit string) and `mcc_description` (lookup, else `UNKNOWN_MCC`).
- `merchant_state` → uppercase (`normalize_state`).
- `use_chip` + `merchant_city` → `is_online` (heuristic).
- Deduping: drop duplicate transaction `id` across chunks and within-chunk.
- Merge sources:
  - join `cards.csv` on `transactions.card_id == cards.id` (PII dropped)
  - join `users.csv` on `transactions.client_id == users.id` (PII dropped)

Assumptions/uncertainties are captured in `artifacts/qa_report.json`.

In [8]:
_CURRENCY_STRIP_RE = re.compile(r"[^0-9\-\.]" )


def clean_text(value: Any) -> str | None:
    """Strip whitespace and collapse internal spaces; returns None for empty/NA."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if not s:
        return None
    s = re.sub(r"\s+", " ", s)
    return s if s else None


def parse_currency_to_float(value: Any) -> float | None:
    """Parse values like "$2,238 " into float. Returns None on failure."""
    s = clean_text(value)
    if s is None:
        return None
    s2 = _CURRENCY_STRIP_RE.sub("", s)
    if s2 in ("", "-", "."):
        return None
    try:
        return float(s2)
    except ValueError:
        return None


def parse_transaction_datetime(value: Any) -> datetime | None:
    """Parse transaction datetime like '2010-01-01 00:07:00'."""
    s = clean_text(value)
    if s is None:
        return None
    try:
        return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None


def normalize_zip(value: Any) -> str | None:
    """Normalize ZIP-like values: '10464.0' -> '10464'."""
    s = clean_text(value)
    if s is None:
        return None
    if s.endswith(".0"):
        s = s[:-2]
    return s


def normalize_mcc(value: Any) -> str | None:
    """Normalize MCC values to digit strings: 5812, '5812.0' -> '5812'."""
    s = clean_text(value)
    if s is None:
        return None
    if s.endswith(".0"):
        s = s[:-2]
    s = s.strip()
    return s if s.isdigit() else None


def normalize_state(value: Any) -> str | None:
    """Normalize state codes to uppercase; returns None if missing."""
    s = clean_text(value)
    if s is None:
        return None
    return s.upper()


def yesno_to_bool(value: Any) -> bool | None:
    s = clean_text(value)
    if s is None:
        return None
    s = s.lower()
    if s in ("yes", "y", "true", "1"):
        return True
    if s in ("no", "n", "false", "0"):
        return False
    return None


def derive_is_online(use_chip: Any, merchant_city: Any) -> bool:
    uc = (clean_text(use_chip) or "").lower()
    mc = (clean_text(merchant_city) or "").lower()
    return ("online" in uc) or (mc == "online")


def dedupe_by_id(df: pd.DataFrame, *, id_col: str, seen_ids: set[str]) -> tuple[pd.DataFrame, int]:
    """Drop duplicate IDs across chunks and within-chunk; returns (deduped_df, dropped_count)."""
    ids = df[id_col].astype(str)
    is_dup = ids.isin(seen_ids)
    dropped = int(is_dup.sum())
    if dropped:
        df = df.loc[~is_dup].copy()
        ids = df[id_col].astype(str)
    before = len(df)
    df = df.drop_duplicates(subset=[id_col], keep="first")
    dropped += before - len(df)
    seen_ids.update(ids.tolist())
    return df, dropped


def append_ndjson(path: Path, df: pd.DataFrame) -> None:
    """Append DataFrame rows to an NDJSON file (one JSON object per line)."""
    # Pandas can write NDJSON with lines=True.
    text = df.to_json(orient="records", lines=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(text)
        if not text.endswith("\n"):
            f.write("\n")


@dataclass
class QaReport:
    transactions_rows_total_scanned: int = 0
    transactions_duplicates_dropped: int = 0
    rows_dropped_invalid_amount: int = 0
    rows_dropped_invalid_date: int = 0
    amount_parse_failures: int = 0
    date_parse_failures: int = 0
    card_join_misses: int = 0
    user_join_misses: int = 0
    mcc_missing_from_lookup: int = 0
    focus_client_id: int = 0
    focus_rows_written: int = 0
    sample_focus_head: list[dict[str, Any]] | None = None
    assumptions: dict[str, str] | None = None


## Load + prepare dimension tables (PII-safe)

Loads `users.csv`, `cards.csv`, and `mcc_codes.json` fully (small) and drops PII columns up front.

In [9]:
for p in PATHS.values():
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

mcc_lookup: dict[str, str] = json.loads(PATHS["mcc"].read_text(encoding="utf-8"))

# Users (drop address; parse monthly limit)
users = pd.read_csv(PATHS["users"], dtype=str)
if "address" in users.columns:
    users = users.drop(columns=["address"])

# Drop user columns not needed for v1 pipeline (per request)
_DROP_USER_COLS = [
    "retirement_age",
    "birth_year",
    "birth_month",
    "gender",
    "latitude",
    "longitude",
]
users = users.drop(columns=[c for c in _DROP_USER_COLS if c in users.columns])
users = users.rename(columns={"id": "client_id"})
users["client_id"] = users["client_id"].map(clean_text)
users["monthly_discretionary_limit_usd"] = users["monthly_discretionary_limits"].map(parse_currency_to_float)

# Cards (drop card_number/cvv; parse credit limit)
cards = pd.read_csv(PATHS["cards"], dtype=str)
for col in ["card_number", "cvv"]:
    if col in cards.columns:
        cards = cards.drop(columns=[col])
cards = cards.rename(columns={"id": "card_id"})
cards["card_id"] = cards["card_id"].map(clean_text)
cards["client_id"] = cards["client_id"].map(clean_text)
cards["has_chip"] = cards["has_chip"].map(yesno_to_bool)
cards["card_on_dark_web"] = cards["card_on_dark_web"].map(yesno_to_bool)
cards["credit_limit_usd"] = cards["credit_limit"].map(parse_currency_to_float)
if "credit_limit" in cards.columns:
    cards = cards.drop(columns=["credit_limit"])

print("users rows:", len(users), "cards rows:", len(cards), "mcc codes:", len(mcc_lookup))


users rows: 100 cards rows: 300 mcc codes: 109


## Stream clean + merge (chunked)

Processes `transactions.csv` in chunks to avoid loading the full dataset.

Writes:
- all users: NDJSON stream to `artifacts/transactions_enriched.json`
- optional: if `FOCUS_CLIENT_ID` is set, also writes `artifacts/transactions_enriched_{client_id}.json`


In [10]:
# Reset output files for reproducible reruns
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
if OUT_ALL_NDJSON.exists():
    OUT_ALL_NDJSON.unlink()
if OUT_FOCUS_JSON is not None and OUT_FOCUS_JSON.exists():
    OUT_FOCUS_JSON.unlink()

qa = QaReport(
    focus_client_id=FOCUS_CLIENT_ID,
    assumptions={
        "datetime_format": "%Y-%m-%d %H:%M:%S",
        "currency_format": "optional $, commas, whitespace; otherwise dropped",
        "dedupe_policy": "transactions.id unique; keep first occurrence",
        "is_online_heuristic": "use_chip contains 'online' OR merchant_city == 'ONLINE'",
        "mcc_normalization": "digit strings; float-like strings ending in .0 are stripped; else UNKNOWN_MCC",
    },
)
seen_tx_ids: set[str] = set()
focus_records: list[dict[str, Any]] = []

usecols = [
    "id",
    "date",
    "client_id",
    "card_id",
    "amount",
    "use_chip",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "zip",
    "mcc",
    "errors",
]

for chunk in pd.read_csv(
    PATHS["transactions"],
    dtype=str,
    usecols=usecols,
    chunksize=TRANSACTIONS_CHUNKSIZE,
):
    qa.transactions_rows_total_scanned += len(chunk)

    # Standardize text fields
    for col in usecols:
        if col in chunk.columns:
            chunk[col] = chunk[col].map(clean_text)
    chunk["merchant_state"] = chunk["merchant_state"].map(normalize_state)

    # Deduplicate
    chunk, dropped = dedupe_by_id(chunk, id_col="id", seen_ids=seen_tx_ids)
    qa.transactions_duplicates_dropped += dropped
    if chunk.empty:
        continue

    # Parse amount + drop invalid
    chunk["amount_usd"] = chunk["amount"].map(parse_currency_to_float)
    qa.amount_parse_failures += int(chunk["amount_usd"].isna().sum())
    invalid_amount = chunk["amount_usd"].isna()
    qa.rows_dropped_invalid_amount += int(invalid_amount.sum())
    if int(invalid_amount.sum()) > 0:
        chunk = chunk.loc[~invalid_amount].copy()
    if chunk.empty:
        continue

    # Parse date + drop invalid
    chunk["transaction_dt"] = chunk["date"].map(parse_transaction_datetime)
    qa.date_parse_failures += int(pd.isna(chunk["transaction_dt"]).sum())
    invalid_date = pd.isna(chunk["transaction_dt"])
    qa.rows_dropped_invalid_date += int(invalid_date.sum())
    if int(invalid_date.sum()) > 0:
        chunk = chunk.loc[~invalid_date].copy()
    if chunk.empty:
        continue

    # Normalize + enrich MCC
    chunk["zip_norm"] = chunk["zip"].map(normalize_zip)
    chunk["mcc_code"] = chunk["mcc"].map(normalize_mcc)
    chunk["mcc_description"] = chunk["mcc_code"].map(
        lambda c: mcc_lookup.get(str(c), "UNKNOWN_MCC") if c else "UNKNOWN_MCC"
    )
    qa.mcc_missing_from_lookup += int((chunk["mcc_description"] == "UNKNOWN_MCC").sum())

    # Heuristic online flag
    chunk["is_online"] = chunk.apply(
        lambda r: derive_is_online(r.get("use_chip"), r.get("merchant_city")),
        axis=1,
    )

    # Merge cards + users
    chunk = chunk.merge(cards, on="card_id", how="left", suffixes=("", "_card"))
    qa.card_join_misses += int(chunk["card_brand"].isna().sum()) if "card_brand" in chunk.columns else 0

    chunk = chunk.merge(users, on="client_id", how="left", suffixes=("", "_user"))
    qa.user_join_misses += int(chunk["monthly_discretionary_limit_usd"].isna().sum())

    # Ensure PII columns are not present (defense-in-depth)
    for pii in ["address", "card_number", "cvv"]:
        if pii in chunk.columns:
            chunk = chunk.drop(columns=[pii])

    # Serialize datetimes to ISO strings for JSON output
    chunk["transaction_dt"] = chunk["transaction_dt"].map(lambda d: d.isoformat(sep=" ") if d is not None and not pd.isna(d) else None)

    # Append all-users NDJSON
    append_ndjson(OUT_ALL_NDJSON, chunk)

    # Optional: capture focus user rows
    if FOCUS_CLIENT_ID is not None:
        focus_chunk = chunk.loc[chunk["client_id"] == str(FOCUS_CLIENT_ID)].copy()
        if not focus_chunk.empty:
            if qa.sample_focus_head is None:
                qa.sample_focus_head = focus_chunk.head(QA_SAMPLE_ROWS).to_dict(orient="records")
            focus_records.extend(focus_chunk.to_dict(orient="records"))

if OUT_FOCUS_JSON is not None:
    # Write focus JSON array
    OUT_FOCUS_JSON.write_text(json.dumps(focus_records, indent=2), encoding="utf-8")
    qa.focus_rows_written = len(focus_records)

# Write QA report
OUT_QA.write_text(json.dumps(asdict(qa), indent=2, sort_keys=True), encoding="utf-8")

print("Wrote:", OUT_ALL_NDJSON)
if OUT_FOCUS_JSON is not None:
    print("Wrote:", OUT_FOCUS_JSON)
print("Wrote:", OUT_QA)
if OUT_FOCUS_JSON is not None:
    print("Focus rows:", qa.focus_rows_written)


Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/transactions_enriched.json
Wrote: /Users/nicholasp/Personal Coding/JHU/personal finance/artifacts/qa_report.json
